# Notebook 4: RAG Pipeline — From Scratch to Production

**Goal:** Build a complete Retrieval-Augmented Generation (RAG) pipeline. RAG is one of the most common LLM system design topics.

**Topics:**
1. Why RAG? (vs fine-tuning, vs long context)
2. Text chunking strategies
3. Embeddings from scratch (cosine similarity, dot product)
4. In-memory vector store
5. Retrieval with ranking
6. Context packing and prompt construction
7. Full RAG pipeline (embed, retrieve, generate)
8. Hybrid search (dense + sparse/BM25)
9. Reranking
10. RAG failure modes and fixes

In [ ]:
import numpy as np
import math
import re
import os
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter

np.random.seed(42)
print('Setup complete')

---
## 1. Why RAG?

| Approach | Pros | Cons | Use When |
|---|---|---|---|
| RAG | No retraining, up-to-date, source-grounded | Latency overhead, retrieval can fail | Knowledge base Q&A, live data |
| Fine-tuning | Learns new behavior/format | Expensive, needs labeled data, can forget | New task format, tone/persona |
| Long context | Simple, no retrieval | Very expensive, needle-in-haystack hard | Small corpora, occasional use |

**Interview answer:** 'RAG is the right default for knowledge injection. Fine-tuning is for behavior change, not knowledge change.'

---
## 2. Text Chunking Strategies

Documents must be split into chunks before embedding. The chunking strategy greatly affects retrieval quality.

In [ ]:
def chunk_fixed_size(
    text: str,
    chunk_size: int = 512,
    overlap: int = 64,
) -> List[str]:
    """
    Split text into fixed-size character chunks with overlap.
    Overlap ensures context at chunk boundaries isn't lost.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


def chunk_by_sentence(text: str, max_chunk_size: int = 512) -> List[str]:
    """
    Split text by sentences, grouping until max_chunk_size chars.
    More semantically coherent than fixed-size splitting.
    """
    # YOUR CODE HERE
    # Step 1: Split into sentences (use regex for sentence boundaries)
    # Step 2: Group sentences until chunk would exceed max_chunk_size
    raise NotImplementedError()


sample_text = """
Mistral AI is a French artificial intelligence company founded in 2023. 
The company focuses on developing and deploying large language models.
Their flagship model, Mistral 7B, uses grouped query attention and sliding window attention.
These techniques make inference faster and more memory efficient.
Mixtral 8x7B is a sparse mixture of experts model that activates only 2 of 8 experts per token.
This architecture achieves strong performance while keeping inference compute manageable.
La Plateforme is Mistral's API service, analogous to OpenAI's platform.
Mistral offers both open-weight models and proprietary closed models through their API.
"""

# chunks = chunk_fixed_size(sample_text, chunk_size=200, overlap=50)
# print(f'Fixed chunks: {len(chunks)}')
# for i, c in enumerate(chunks):
#     print(f'[Chunk {i}] {c[:80]}...')

In [ ]:
# REFERENCE
def chunk_fixed_size_ref(text: str, chunk_size: int = 512, overlap: int = 64) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # move forward by (chunk_size - overlap)
    return [c.strip() for c in chunks if c.strip()]


def chunk_by_sentence_ref(text: str, max_chunk_size: int = 512) -> List[str]:
    # Split by sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    chunks = []
    current_chunk = ''
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        if len(current_chunk) + len(sentence) + 1 <= max_chunk_size:
            current_chunk = (current_chunk + ' ' + sentence).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks


chunk_fixed_size = chunk_fixed_size_ref
chunk_by_sentence = chunk_by_sentence_ref

fixed_chunks = chunk_fixed_size(sample_text, chunk_size=200, overlap=50)
print(f'Fixed-size chunks ({len(fixed_chunks)}):')
for i, c in enumerate(fixed_chunks):
    print(f'  [{i}] {c[:70].strip()}...')

print(f'\nSentence chunks ({len(chunk_by_sentence(sample_text))})')

---
## 3. Similarity Metrics

Three common ways to compare embeddings:
- **Cosine similarity:** angle between vectors, scale-invariant (most common)
- **Dot product:** magnitude + angle (used when embeddings are normalized)
- **L2 distance:** Euclidean distance (lower = more similar)

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity: dot(a,b) / (|a| * |b|). Range [-1, 1]."""
    # YOUR CODE HERE
    raise NotImplementedError()


def cosine_similarity_batch(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between a query vector and all rows of a matrix.
    
    Args:
        query: (d,) vector
        matrix: (N, d) matrix of N vectors
    Returns:
        (N,) similarity scores
    """
    # YOUR CODE HERE
    # Hint: normalize both, then matrix multiply
    raise NotImplementedError()


def l2_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Euclidean distance."""
    # YOUR CODE HERE
    raise NotImplementedError()


# Test: identical vectors should have similarity 1
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([2.0, 4.0, 6.0])  # same direction, different magnitude
v3 = np.array([-1.0, -2.0, -3.0])  # opposite direction
print('v1 vs v2 (same direction):', cosine_similarity(v1, v2))  # should be ~1.0
print('v1 vs v3 (opposite):', cosine_similarity(v1, v3))  # should be ~-1.0

In [ ]:
# REFERENCE
def cosine_similarity_ref(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def cosine_similarity_batch_ref(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    # Normalize
    query_norm = query / (np.linalg.norm(query) + 1e-8)
    matrix_norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-8
    matrix_normalized = matrix / matrix_norms
    # Dot product with normalized vectors = cosine similarity
    return matrix_normalized @ query_norm


def l2_distance_ref(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


cosine_similarity = cosine_similarity_ref
cosine_similarity_batch = cosine_similarity_batch_ref
l2_distance = l2_distance_ref

v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([2.0, 4.0, 6.0])
v3 = np.array([-1.0, -2.0, -3.0])
print('v1 vs v2 (same direction):', cosine_similarity(v1, v2))  # ~1.0
print('v1 vs v3 (opposite):', cosine_similarity(v1, v3))  # ~-1.0

# Batch test
matrix = np.array([[1, 2, 3], [4, 5, 6], [-1, -2, -3]], dtype=float)
sims = cosine_similarity_batch(v1, matrix)
print('Batch similarities:', sims.round(3))

---
## 4. In-Memory Vector Store

In [ ]:
@dataclass
class Document:
    id: str
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None


class VectorStore:
    """
    Simple in-memory vector store with exact cosine similarity search.
    For production: use FAISS, Chroma, Pinecone, Weaviate, etc.
    """
    
    def __init__(self):
        self.documents: List[Document] = []
        self._embeddings: Optional[np.ndarray] = None  # cached matrix for fast search
    
    def add(self, doc: Document) -> None:
        """Add a document (must have embedding set)."""
        # YOUR CODE HERE
        # Add to self.documents, invalidate cache
        raise NotImplementedError()
    
    def add_many(self, docs: List[Document]) -> None:
        """Add multiple documents efficiently."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def _get_embedding_matrix(self) -> np.ndarray:
        """Build and cache the (N, d) embedding matrix."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int = 5,
        score_threshold: float = 0.0,
    ) -> List[Tuple[Document, float]]:
        """
        Find top_k most similar documents by cosine similarity.
        
        Returns: list of (document, score) tuples, sorted by score descending.
        """
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def __len__(self) -> int:
        return len(self.documents)

print('VectorStore defined — implement methods above')

In [ ]:
# REFERENCE
class VectorStoreRef:
    def __init__(self):
        self.documents: List[Document] = []
        self._embeddings: Optional[np.ndarray] = None
    
    def add(self, doc: Document) -> None:
        assert doc.embedding is not None, 'Document must have embedding'
        self.documents.append(doc)
        self._embeddings = None  # invalidate cache
    
    def add_many(self, docs: List[Document]) -> None:
        for doc in docs:
            assert doc.embedding is not None
            self.documents.append(doc)
        self._embeddings = None
    
    def _get_embedding_matrix(self) -> np.ndarray:
        if self._embeddings is None:
            self._embeddings = np.stack([d.embedding for d in self.documents])
        return self._embeddings
    
    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int = 5,
        score_threshold: float = 0.0,
    ) -> List[Tuple[Document, float]]:
        if not self.documents:
            return []
        
        matrix = self._get_embedding_matrix()
        scores = cosine_similarity_batch(query_embedding, matrix)
        
        # Get top_k indices sorted by score
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            score = float(scores[idx])
            if score >= score_threshold:
                results.append((self.documents[idx], score))
        
        return results
    
    def __len__(self) -> int:
        return len(self.documents)


VectorStore = VectorStoreRef

# Test with fake embeddings
store = VectorStore()
np.random.seed(42)

# Create docs with random embeddings (in real life: use embedding model)
topics = [
    ('attention', 'Self-attention computes QKV matrices.'),
    ('moe', 'Mixture of Experts uses sparse gating.'),
    ('rag', 'RAG combines retrieval with generation.'),
    ('lora', 'LoRA injects low-rank adapters into weights.'),
    ('kv_cache', 'KV cache stores key-value pairs for autoregressive generation.'),
]

for doc_id, text in topics:
    # In real RAG: embedding = embed_model.encode(text)
    fake_embedding = np.random.randn(128)
    store.add(Document(id=doc_id, text=text, embedding=fake_embedding))

query_emb = np.random.randn(128)  # would be embedding of the question
results = store.search(query_emb, top_k=3)
print(f'Search results (top 3):')
for doc, score in results:
    print(f'  [{score:.3f}] {doc.id}: {doc.text[:50]}')

---
## 5. BM25 Sparse Retrieval (for Hybrid Search)

In [ ]:
class BM25:
    """
    BM25 (Best Match 25) — classic sparse retrieval.
    Better than TF-IDF for keyword matching.
    
    Score(D, Q) = sum_t [ IDF(t) * TF(t,D) * (k1+1) / (TF(t,D) + k1*(1-b+b*|D|/avgdl)) ]
    """
    
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs: List[List[str]] = []
        self.idf: Dict[str, float] = {}
        self.avgdl: float = 0.0
    
    def tokenize(self, text: str) -> List[str]:
        """Simple whitespace tokenizer."""
        return text.lower().split()
    
    def fit(self, documents: List[str]) -> 'BM25':
        """
        Build index from a list of document strings.
        """
        # YOUR CODE HERE
        # 1. Tokenize each document
        # 2. Compute avgdl
        # 3. Compute IDF for each term
        # IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1)
        # where N = total docs, df(t) = docs containing t
        raise NotImplementedError()
    
    def score(self, query: str, doc_idx: int) -> float:
        """BM25 score for a single document."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Return top_k (doc_index, score) pairs."""
        # YOUR CODE HERE
        raise NotImplementedError()

# Test
bm25_docs = [doc.text for _, doc in enumerate(store.documents)]
# bm25 = BM25().fit(bm25_docs)
# results = bm25.search('attention key value', top_k=3)
# for idx, score in results:
#     print(f'[{score:.3f}] {bm25_docs[idx][:60]}')

In [ ]:
# REFERENCE
class BM25Ref:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs: List[List[str]] = []
        self.idf: Dict[str, float] = {}
        self.avgdl: float = 0.0
    
    def tokenize(self, text: str) -> List[str]:
        return re.findall(r'\w+', text.lower())
    
    def fit(self, documents: List[str]) -> 'BM25Ref':
        self.docs = [self.tokenize(d) for d in documents]
        N = len(self.docs)
        self.avgdl = np.mean([len(d) for d in self.docs])
        
        # Document frequency
        df: Counter = Counter()
        for doc in self.docs:
            df.update(set(doc))
        
        # IDF with smoothing
        self.idf = {}
        for term, doc_freq in df.items():
            self.idf[term] = math.log((N - doc_freq + 0.5) / (doc_freq + 0.5) + 1)
        
        return self
    
    def score(self, query: str, doc_idx: int) -> float:
        query_tokens = self.tokenize(query)
        doc = self.docs[doc_idx]
        doc_len = len(doc)
        tf = Counter(doc)
        
        total = 0.0
        for term in query_tokens:
            if term not in self.idf:
                continue
            tf_t = tf.get(term, 0)
            numerator = self.idf[term] * tf_t * (self.k1 + 1)
            denominator = tf_t + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            total += numerator / denominator
        
        return total
    
    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        scores = [(i, self.score(query, i)) for i in range(len(self.docs))]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]


BM25 = BM25Ref

bm25_docs = [doc.text for doc in store.documents]
bm25 = BM25().fit(bm25_docs)
results = bm25.search('attention key value cache', top_k=3)
print('BM25 results for "attention key value cache":')
for idx, score in results:
    print(f'  [{score:.3f}] {bm25_docs[idx]}')

---
## 6. Hybrid Search (Dense + Sparse)

In [ ]:
def hybrid_search(
    query: str,
    query_embedding: np.ndarray,
    vector_store: VectorStore,
    bm25_index: BM25,
    top_k: int = 5,
    dense_weight: float = 0.7,
    sparse_weight: float = 0.3,
) -> List[Tuple[Document, float]]:
    """
    Combine dense (embedding) and sparse (BM25) retrieval scores.
    
    Strategy: Reciprocal Rank Fusion (RRF) or weighted score combination.
    Here we use weighted normalized scores.
    """
    # YOUR CODE HERE
    # 1. Get dense scores for all docs
    # 2. Get BM25 scores for all docs
    # 3. Normalize each to [0, 1]
    # 4. Combine: combined = dense_weight * dense + sparse_weight * sparse
    # 5. Return top_k docs sorted by combined score
    raise NotImplementedError()

# REFERENCE
def hybrid_search_ref(
    query: str,
    query_embedding: np.ndarray,
    vector_store: VectorStore,
    bm25_index: BM25,
    top_k: int = 5,
    dense_weight: float = 0.7,
    sparse_weight: float = 0.3,
) -> List[Tuple[Document, float]]:
    n = len(vector_store)
    
    # Dense scores
    matrix = vector_store._get_embedding_matrix()
    dense_scores = cosine_similarity_batch(query_embedding, matrix)
    # Normalize to [0, 1]
    d_min, d_max = dense_scores.min(), dense_scores.max()
    dense_norm = (dense_scores - d_min) / (d_max - d_min + 1e-8)
    
    # Sparse scores
    sparse_raw = np.array([bm25_index.score(query, i) for i in range(n)])
    s_min, s_max = sparse_raw.min(), sparse_raw.max()
    sparse_norm = (sparse_raw - s_min) / (s_max - s_min + 1e-8)
    
    # Combine
    combined = dense_weight * dense_norm + sparse_weight * sparse_norm
    
    top_indices = np.argsort(combined)[::-1][:top_k]
    return [(vector_store.documents[i], float(combined[i])) for i in top_indices]


hybrid_search = hybrid_search_ref

results = hybrid_search(
    query='attention key value',
    query_embedding=np.random.randn(128),
    vector_store=store,
    bm25_index=bm25,
    top_k=3,
)
print('Hybrid search results:')
for doc, score in results:
    print(f'  [{score:.3f}] {doc.text[:60]}')

---
## 7. Full RAG Pipeline

In [ ]:
class RAGPipeline:
    """
    End-to-end RAG pipeline:
    Document -> Chunk -> Embed -> Store -> Retrieve -> Generate
    """
    
    def __init__(
        self,
        embed_fn,        # Callable: text -> np.ndarray
        generate_fn,     # Callable: (question, contexts) -> str
        chunk_size: int = 512,
        chunk_overlap: int = 64,
        top_k: int = 5,
    ):
        self.embed_fn = embed_fn
        self.generate_fn = generate_fn
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        self.vector_store = VectorStore()
    
    def add_document(
        self,
        text: str,
        doc_id: str,
        metadata: Dict[str, Any] = None,
    ) -> int:
        """
        Chunk, embed, and store a document.
        Returns number of chunks added.
        """
        # YOUR CODE HERE
        # 1. Chunk the text
        # 2. Embed each chunk
        # 3. Create Document objects with metadata
        # 4. Add to vector_store
        raise NotImplementedError()
    
    def query(
        self,
        question: str,
        return_context: bool = False,
    ) -> str | Dict:
        """
        Answer a question using RAG.
        """
        # YOUR CODE HERE
        # 1. Embed the question
        # 2. Retrieve top-k chunks
        # 3. Build context string from chunks
        # 4. Call generate_fn with question + context
        raise NotImplementedError()

print('RAGPipeline defined')

In [ ]:
# REFERENCE + test with mock functions
class RAGPipelineRef:
    def __init__(self, embed_fn, generate_fn, chunk_size=512, chunk_overlap=64, top_k=5):
        self.embed_fn = embed_fn
        self.generate_fn = generate_fn
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        self.vector_store = VectorStore()
    
    def add_document(self, text: str, doc_id: str, metadata: Dict = None) -> int:
        chunks = chunk_fixed_size(text, self.chunk_size, self.chunk_overlap)
        
        docs = []
        for i, chunk_text in enumerate(chunks):
            embedding = self.embed_fn(chunk_text)
            doc = Document(
                id=f'{doc_id}_chunk_{i}',
                text=chunk_text,
                metadata={'source': doc_id, 'chunk_idx': i, **(metadata or {})},
                embedding=embedding,
            )
            docs.append(doc)
        
        self.vector_store.add_many(docs)
        return len(chunks)
    
    def query(self, question: str, return_context: bool = False):
        q_embedding = self.embed_fn(question)
        results = self.vector_store.search(q_embedding, top_k=self.top_k)
        
        context_pieces = [doc.text for doc, _ in results]
        answer = self.generate_fn(question, context_pieces)
        
        if return_context:
            return {'answer': answer, 'sources': [(doc.id, score) for doc, score in results]}
        return answer


RAGPipeline = RAGPipelineRef

# Test with mock functions
def mock_embed(text: str) -> np.ndarray:
    """Fake embedding: deterministic based on text hash."""
    np.random.seed(hash(text) % 2**31)
    return np.random.randn(64)

def mock_generate(question: str, contexts: List[str]) -> str:
    """Mock generator: just formats the context."""
    context_str = '\n'.join(f'- {c}' for c in contexts[:3])
    return f'[Mock answer to: {question[:30]}]\nBased on:\n{context_str}'

rag = RAGPipeline(embed_fn=mock_embed, generate_fn=mock_generate, top_k=3)

# Index the Mistral knowledge base
mistral_docs = [
    ('mistral_arch', 'Mistral 7B uses sliding window attention with window size 4096. It also uses grouped query attention with 8 KV heads and 32 query heads. The context length is 32k tokens.'),
    ('mixtral', 'Mixtral 8x7B is a sparse mixture of experts model. It has 8 expert FFN layers per block, and each token is routed to 2 experts. The router is a linear layer that outputs 8 logits, then top-2 are selected.'),
    ('training', 'Mistral models are trained using the next-token prediction objective on large text corpora. Pre-training uses bfloat16 precision. The optimizer is AdamW with cosine learning rate schedule.'),
]

for doc_id, text in mistral_docs:
    n = rag.add_document(text, doc_id)
    print(f'Added {doc_id}: {n} chunks')

print(f'\nTotal chunks in store: {len(rag.vector_store)}')

result = rag.query('How does Mixtral route tokens to experts?', return_context=True)
print(f'\nQuestion: How does Mixtral route tokens?')
print(f'Answer: {result["answer"]}')
print(f'Sources: {result["sources"]}')

---
## 8. Prompt Construction for RAG

In [ ]:
def build_rag_prompt(
    question: str,
    contexts: List[str],
    max_context_chars: int = 4000,
) -> str:
    """
    Build a RAG prompt with retrieved context.
    Truncates context if too long.
    """
    # YOUR CODE HERE
    # Build a prompt like:
    # "Use the following context to answer the question..."
    # Context: ...
    # Question: ...
    # Answer:
    raise NotImplementedError()


# REFERENCE
def build_rag_prompt_ref(
    question: str,
    contexts: List[str],
    max_context_chars: int = 4000,
) -> str:
    context_str = ''
    for i, ctx in enumerate(contexts):
        candidate = f'[{i+1}] {ctx}\n'
        if len(context_str) + len(candidate) > max_context_chars:
            break
        context_str += candidate
    
    return f"""Use the following context to answer the question. If the answer is not in the context, say "I don't know based on the provided context."

Context:
{context_str.strip()}

Question: {question}

Answer:"""


build_rag_prompt = build_rag_prompt_ref

prompt = build_rag_prompt(
    'How many experts does Mixtral use per token?',
    ['Mixtral 8x7B activates 2 of 8 experts per token.', 'The router selects top-2 experts.']
)
print(prompt)

---
## 9. RAG Failure Modes — Know These for Interview

| Failure Mode | Symptom | Fix |
|---|---|---|
| Retrieval misses | Wrong chunks retrieved | Better chunking, hybrid search, reranking |
| Context too long | Model ignores context | Fewer chunks, summarization, reranking |
| Hallucination on context | Wrong facts despite good retrieval | Stricter prompting, confidence scoring |
| Semantic gap | Keyword vs meaning mismatch | Dense retrieval, better embeddings |
| Chunk boundary | Answer split across chunks | Overlap, sentence-aware chunking |
| Stale knowledge | Retrieval returns outdated doc | Timestamps in metadata, freshness reranking |

In [ ]:
# EXERCISE: Implement a simple reranker
def simple_rerank(
    query: str,
    candidates: List[Tuple[Document, float]],
    top_k: int = 3,
) -> List[Tuple[Document, float]]:
    """
    Simple keyword overlap reranker.
    In production: use a cross-encoder model.
    
    Score = (keyword overlap tokens) / (total query tokens)
    Final score = 0.5 * retrieval_score + 0.5 * rerank_score
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def simple_rerank_ref(
    query: str,
    candidates: List[Tuple[Document, float]],
    top_k: int = 3,
) -> List[Tuple[Document, float]]:
    query_tokens = set(re.findall(r'\w+', query.lower()))
    
    reranked = []
    for doc, retrieval_score in candidates:
        doc_tokens = set(re.findall(r'\w+', doc.text.lower()))
        overlap = len(query_tokens & doc_tokens) / (len(query_tokens) + 1e-8)
        combined = 0.5 * retrieval_score + 0.5 * overlap
        reranked.append((doc, combined))
    
    return sorted(reranked, key=lambda x: x[1], reverse=True)[:top_k]


simple_rerank = simple_rerank_ref
print('Reranker implemented')

---
## 10. Interview Questions

1. **System design:** Design a RAG system for 1M legal documents with sub-500ms P95 latency.

2. **Chunking:** Why does chunk size matter? What happens if chunks are too large? Too small?

3. **Hybrid:** When would you prefer BM25 over dense retrieval? (Rare terms, entity names, exact strings)

4. **Reranking:** What is a cross-encoder reranker? When is the extra latency worth it?

5. **RAG vs fine-tuning:** A customer wants the model to 'know' their product catalog. Which approach?

6. **Context window:** How many chunks can you fit in Mistral 7B's 32k context? (Assume 200-word chunks, ~250 tokens each → about 128 chunks)

---
**Next:** `05_llm_optimization_concepts.ipynb`